# Catan Player Profiling
This notebook aggregates the game summaries and creates a dataframe profiling each player with various statistics.

In [1]:
import polars as pl
from pathlib import Path

# 1. Aggregate all the results across different games
out_dir = Path('../../data/game_log_dfs')
parquet_files = list(out_dir.glob('*.parquet'))

dfs = []
for p in parquet_files:
    # If the file hasn't been re-generated yet, it might not exist if they ran csv logic instead
    # However we added the parquet generator previously
    # Extract game id from "game_{game_id}_stats.parquet"
    game_id = p.name.split('_')[1]
    try:
        df_game = pl.read_parquet(p)
        df_game = df_game.with_columns(pl.lit(game_id).alias("game_id"))
        dfs.append(df_game)
    except Exception as e:
        print(f"Error loading {p}: {e}")

if dfs:
    df_all = pl.concat(dfs, how="diagonal")
    print(f"Aggregated {len(dfs)} games.")


Aggregated 67 games.


In [2]:
if not df_all.is_empty():
    # Helper to calculate metrics per player
    
    # First calculate the flat player aggs
    player_profiles = df_all.group_by('player').agg(
        num_games=pl.len(),
        avg_turns_before_first_trade=pl.col('turns_before_first_trade').filter(pl.col('turns_before_first_trade') != -1).mean() - 9,
        ratio_cards_given_to_taken=(pl.col('cards_lost_in_trades').sum() / pl.col('cards_gained_in_trades').sum()),
        trade_success_rate=(pl.col('trades_completed').sum() / pl.col('trades_proposed').sum()),
        avg_counter_offers=pl.col('counter_offers').mean(),
        avg_bank_trades=pl.col('bank_trades').mean(),
        avg_dev_cards_bought=pl.col('dev_cards_bought').mean(),
        avg_roads_built=pl.col('roads_built').mean(),
        avg_cities_built=pl.col('cities_built').mean(),
        avg_players_targeted=pl.col('unique_players_stolen_from').mean(),
        avg_times_targeted=pl.col('times_targeted_by_others').mean(),
        win_rate_largest_army=(pl.col('largest_army_received') > 0).sum() / pl.len(),
        win_rate_longest_road=(pl.col('longest_road_received') > 0).sum() / pl.len(),
        
        average_games_with_port_used=pl.col('port_used').sum() / pl.len(),
        
        # NEW: Hand Size Trend
        overall_avg_hand_size=pl.col('avg_hand_size').mean(),
        
        # We need these to calculate top 3 starting resources per player
        total_starting_ore=pl.col('starting_ore').fill_null(0).sum(),
        total_starting_wool=pl.col('starting_wool').fill_null(0).sum(),
        total_starting_lumber=pl.col('starting_lumber').fill_null(0).sum(),
        total_starting_grain=pl.col('starting_grain').fill_null(0).sum(),
        total_starting_brick=pl.col('starting_brick').fill_null(0).sum(),
        
        # NEW: Resource Scarcity additions for trades
        total_traded_away_ore=pl.col('traded_away_ore').fill_null(0).sum(),
        total_traded_away_wool=pl.col('traded_away_wool').fill_null(0).sum(),
        total_traded_away_lumber=pl.col('traded_away_lumber').fill_null(0).sum(),
        total_traded_away_grain=pl.col('traded_away_grain').fill_null(0).sum(),
        total_traded_away_brick=pl.col('traded_away_brick').fill_null(0).sum(),
        
        total_received_trade_ore=pl.col('received_trade_ore').fill_null(0).sum(),
        total_received_trade_wool=pl.col('received_trade_wool').fill_null(0).sum(),
        total_received_trade_lumber=pl.col('received_trade_lumber').fill_null(0).sum(),
        total_received_trade_grain=pl.col('received_trade_grain').fill_null(0).sum(),
        total_received_trade_brick=pl.col('received_trade_brick').fill_null(0).sum()
    )
    
    # Calculate top 3 starting resources and trading behaviors
    derived_metrics_list = []
    
    for row in player_profiles.iter_rows(named=True):
        player = row['player']
        
        # Starting resources
        res_counts = {
            'ore': row['total_starting_ore'],
            'wool': row['total_starting_wool'],
            'lumber': row['total_starting_lumber'],
            'grain': row['total_starting_grain'],
            'brick': row['total_starting_brick']
        }
        sorted_res = sorted(res_counts.items(), key=lambda item: item[1], reverse=True)
        top_3 = [res for res, count in sorted_res[:3]]
        
        # Traded away stats
        traded_away = {
            'ore': row['total_traded_away_ore'],
            'wool': row['total_traded_away_wool'],
            'lumber': row['total_traded_away_lumber'],
            'grain': row['total_traded_away_grain'],
            'brick': row['total_traded_away_brick']
        }
        most_traded_away = max(traded_away, key=traded_away.get) if sum(traded_away.values()) > 0 else "None"
        
        # Received stats
        received_trade = {
            'ore': row['total_received_trade_ore'],
            'wool': row['total_received_trade_wool'],
            'lumber': row['total_received_trade_lumber'],
            'grain': row['total_received_trade_grain'],
            'brick': row['total_received_trade_brick']
        }
        most_received = max(received_trade, key=received_trade.get) if sum(received_trade.values()) > 0 else "None"
        
        derived_metrics_list.append({
            'player': player,
            'top_3_starting_resources': ", ".join(top_3),
            'most_traded_away_resource': most_traded_away,
            'most_received_resource': most_received
        })
    
    df_derived = pl.DataFrame(derived_metrics_list)
    
    # Join the derived columns back 
    player_profiles = player_profiles.join(df_derived, on='player', how='left')
    
    # Drop the intermediate aggregate columns to clean up the final profiled DataFrame
    columns_to_drop = [
        'total_starting_ore', 'total_starting_wool', 'total_starting_lumber', 
        'total_starting_grain', 'total_starting_brick',
        'total_traded_away_ore', 'total_traded_away_wool', 'total_traded_away_lumber', 
        'total_traded_away_grain', 'total_traded_away_brick',
        'total_received_trade_ore', 'total_received_trade_wool', 'total_received_trade_lumber', 
        'total_received_trade_grain', 'total_received_trade_brick'
    ]
    player_profiles = player_profiles.drop(columns_to_drop)
    
    # Sort by number of games played (descending)
    player_profiles = player_profiles.sort('num_games', descending=True)
    
    # Display the result
    print("PLAYER PROFILES:")
    display(player_profiles.head(9))
else:
    print("Could not generate player profiles because df_all is empty.")


PLAYER PROFILES:


player,num_games,avg_turns_before_first_trade,ratio_cards_given_to_taken,trade_success_rate,avg_counter_offers,avg_bank_trades,avg_dev_cards_bought,avg_roads_built,avg_cities_built,avg_players_targeted,avg_times_targeted,win_rate_largest_army,win_rate_longest_road,average_games_with_port_used,overall_avg_hand_size,top_3_starting_resources,most_traded_away_resource,most_received_resource
str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""HomeofAD3005""",18,9.777778,0.953125,0.186567,3.166667,5.611111,3.0,7.444444,1.722222,2.111111,0.0,0.111111,0.388889,1.0,6.87666,"""lumber, grain, brick""","""wool""","""ore"""
"""AatNeverLose""",17,14.529412,0.901408,0.226804,0.0,3.882353,2.647059,6.882353,1.705882,1.647059,0.0,0.235294,0.117647,0.882353,8.207652,"""brick, lumber, grain""","""wool""","""grain"""
"""Romeoore""",13,10.692308,0.986486,0.183824,2.384615,3.538462,6.0,6.384615,1.846154,2.461538,0.0,0.538462,0.153846,0.692308,6.737032,"""grain, wool, ore""","""wool""","""ore"""
"""ZL24""",12,10.166667,1.209677,0.328358,0.916667,4.75,4.083333,7.0,2.75,0.25,1.916667,0.333333,0.083333,1.0,6.023109,"""lumber, wool, grain""","""wool""","""ore"""
"""UniQueLagacy""",10,7.0,1.066667,0.189655,2.7,4.7,2.2,7.8,1.5,1.6,0.0,0.0,0.4,0.8,8.375251,"""grain, lumber, brick""","""lumber""","""brick"""
"""GoldenGrain""",3,18.333333,0.866667,0.333333,1.0,6.333333,3.0,7.0,1.333333,1.333333,5.333333,0.0,0.0,1.0,7.058081,"""grain, lumber, wool""","""wool""","""grain"""
"""Deleted User""",3,16.333333,1.222222,0.25,0.333333,4.0,3.333333,6.333333,1.666667,1.333333,4.0,0.0,0.333333,1.0,6.166014,"""wool, grain, ore""","""wool""","""grain"""
"""Ehr4840""",2,36.5,1.5,0.5,0.0,4.5,2.5,6.0,0.5,1.0,2.5,0.0,0.5,0.5,4.007654,"""wool, lumber, grain""","""lumber""","""ore"""
"""SirBeef""",2,4.0,0.625,0.230769,0.0,3.0,6.5,5.5,1.0,1.5,3.0,1.0,0.0,1.0,5.905195,"""wool, lumber, ore""","""wool""","""grain"""


In [3]:
# Filter for players with at least 4 games
player_profiles_filtered = player_profiles.filter(pl.col('num_games') >= 4)

# Create the output directory if it doesn't exist
output_dir = Path('../../data/player_profiles')
output_dir.mkdir(parents=True, exist_ok=True)

# Save the filtered player profiles dataframe to parquet
output_path = output_dir / 'player_profiles.parquet'
player_profiles_filtered.write_parquet(output_path)

print(f"Player profiles exported successfully to {output_path.resolve()}")
print(f"Total players with at least 4 games: {len(player_profiles_filtered)}")
print(f"(Filtered from {len(player_profiles)} total players)")


Player profiles exported successfully to /Users/aatmiya/Documents/GitHub/Redeem-Catan/data/player_profiles/player_profiles.parquet
Total players with at least 4 games: 5
(Filtered from 193 total players)
